## <center> Практическое домашнее задание 3 </center>

*`credits to ML course by E. Sokolov`*

### Общая информация

Дата выдачи: 06.04.2026

Мягкий дедлайн: 07:59MSK 25.04.2026

Жесткий дедлайн: 07:59MSK 28.04.2026

### Оглавление

В этом задании вы:

1. [ознакомитесь с тем, что происходит "внутри" метода опорных векторов и логистической регрессии](#введение)
2. [познакомитесь с калибровкой вероятности](#калибровка)
3. [изучите методы трансформации переменных и методы отбора признаков](#кодировка)
4. [попробуете оценить экономический эффект модели](#модель)

### Оценивание и штрафы
Каждая из задач имеет определенную «стоимость» (указана в скобках около задачи). Проверяющий имеет право снизить оценку за неэффективную реализацию или неопрятные графики.

**Обратите внимание**, что в каждом разделе домашнего задания есть оцениваниемые задачи и есть вопросы. Вопросы дополняют задачи и направлены на то, чтобы проинтерпретировать или обосновать происходящее. Код без интерпретации не имеет смысла, поэтому отвечать на вопросы обязательно — за отсутствие ответов мы будем снижать баллы за задачи. Если вы ответите на вопросы, но не напишете корректный код к соответствующим оцениваемым задачам, то баллы за такое выставлены не будут.

**Штраф за каждый день просрочки мягкого дедлайна**: 0.5 баллов. После жесткого дедлайна работы не принимаются.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. 

❗️ Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, необходимо **указать ссылку на этот источник**.

**❗️ При использовании генеративных нейросетей:**

1. Необходимо указывать их как источник в каждом соответствующем задании
2. Ассистенты могут проводить мини-защиты ДЗ
3. Без указания как источника и подозрениях ассистентов, задания будут так же защищаться, иначе обнуляться.

# Часть 1. SVM, LR и калибровка вероятностей<a name="введение"></a> 

*Максимум: 3 балла*

In [ ]:
import os
import time
import warnings
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.compose import ColumnTransformer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    auc,
    average_precision_score,
    make_scorer,
    precision_recall_curve,
    PrecisionRecallDisplay,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import LinearSVC, SVC

warnings.filterwarnings("ignore")

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

def make_ohe():
    """OneHotEncoder с совместимостью для разных версий sklearn."""
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


#### __Задание 1.1  Сравнение методов__ (1 балл)



Сгенерируем синтетические данные.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# фиксируем random_state для воспроизводимости результатов
X, y = make_classification(
    n_samples=10000, n_features=10, n_informative=5, n_redundant=5, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
y_train.mean()

__Случайный классификатор__

Для начала зададим самую простую модель, которая на каждом объекте выдаёт случайный ответ. По тестовой выборке вычислим AUC-ROC, AUC-PR.

In [ ]:
from sklearn.dummy import DummyClassifier


random_classifier = DummyClassifier(strategy='uniform', random_state=42).fit(X_train, y_train)
y_random = random_classifier.predict_proba(X_test)[:,1]
y_random

**Ваш Ответ**: `y_random` — это не итоговый класс, а оценка вероятности положительного класса, которую возвращает `predict_proba`. Для ROC/PR-кривых как раз нужны не только метки 0/1, а непрерывные скоринговые значения, чтобы можно было менять порог классификации.


*Ниже приведен **пример** работы* со встроенными функциями `sklearn` для отрисовки ROC и PR кривых, сохранения метрик. Пайплайн можно изменять как вам удобно.

In [ ]:
from sklearn.metrics import auc
from sklearn.metrics import average_precision_score
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import PrecisionRecallDisplay

from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay

In [ ]:
def depict_pr_roc(y_true, y_pred, classifier_name='Some Classifier', ax=None):
    if ax is None:
        fig, ax = plt.subplots(1, 2, figsize=(11, 5))

    print(classifier_name, 'metrics')
    PrecisionRecallDisplay.from_predictions(y_true, y_pred, ax=ax[0], name=classifier_name)
    precision, recall, _ = precision_recall_curve(y_true, y_pred)
    print('AUC-PR: %.5f' % auc(recall, precision))
    ax[0].set_title("PRC")
    ax[0].set_ylim(0, 1.1)
    RocCurveDisplay.from_predictions(y_true, y_pred, ax=ax[1], name=classifier_name)
    print('AUC-ROC: %.5f' % roc_auc_score(y_true, y_pred))
    ax[1].set_title("ROC")
    ax[1].set_ylim(0, 1.1)
    
    plt.tight_layout()
    plt.legend()


depict_pr_roc(y_test, y_random, 'Random Classifier')

In [ ]:
# DataFrame для сравнения методов классификации по метрикам
df_metrics = pd.DataFrame(columns=['AUC_PR', 'AUC_ROC', 'reg_const'])
precision, recall, _ = precision_recall_curve(y_test, y_random)

# Добавление очередной строки с характеристиками метода
df_metrics.loc['Random Classifier'] = [
      auc(recall, precision),
      roc_auc_score(y_test, y_random),
      0
]

# по аналогии результаты следующих экспериментов можно будет собрать в табличку
df_metrics

__Support Vector Machine (Linear Kernel)__

Обучите метод опорных векторов.

На занятиях мы проходили линейный вариант без ядрового перехода, что соответствует линейному ядру (`LinearSVC` или же `SVC(kernel='linear')` в `sklearn.svm`).

Подберите параметр регуляризации `C` с точки зрения AUC-PR (можете воспользоваться кросс-валидацией или отделить валидационную выборку от обучающей).


In [ ]:
C_value = np.logspace(-3, 2, 6)

svc_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearSVC(max_iter=5000, random_state=42))
])

C_searcher = GridSearchCV(
    svc_pipeline,
    param_grid={'model__C': C_value},
    scoring='average_precision',
    cv=3,
    n_jobs=-1
)

C_searcher.fit(X_train, y_train)

best_C = C_searcher.best_params_['model__C']
svc_pred = C_searcher.decision_function(X_test)

print("Best C value = %.4f" % best_C)
print('SVC AUC-PR:', average_precision_score(y_test, svc_pred))


  На тестовой части:
  - постройте ROC и PR кривые,
  - посчитайте AUC-ROC, AUC-PR с точностью до 5 знаков

In [ ]:
depict_pr_roc(y_test, svc_pred, 'Linear SVM')


Сравните AUC-ROC и AUC-PR для вашей модели с этими же метриками для случайного классификатора.

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, svc_pred)

df_metrics.loc['Linear SVM'] = [
    auc(recall, precision),
    roc_auc_score(y_test, svc_pred),
    best_C
]

df_metrics


__Logistic Regression__


Аналогичное задание для логистической регрессии с L2 регуляризатором:


*   подберите гиперпараметр C, используя метрику AUC-PR
*   нарисуйте ROC, PR кривые для тестовой части
*   выведите метрики для тестовых данных и сравните их с результатами случайного классификатора



In [ ]:
C_value = np.logspace(-3, 2, 6)

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=3000, random_state=42))
])

lr_searcher = GridSearchCV(
    lr_pipeline,
    param_grid={'model__C': C_value},
    scoring='average_precision',
    cv=3,
    n_jobs=-1
)

lr_searcher.fit(X_train, y_train)

best_lr_C = lr_searcher.best_params_['model__C']
lr_pred = lr_searcher.predict_proba(X_test)[:, 1]

depict_pr_roc(y_test, lr_pred, 'Logistic Regression')

precision, recall, _ = precision_recall_curve(y_test, lr_pred)
df_metrics.loc['Logistic Regression'] = [
    auc(recall, precision),
    roc_auc_score(y_test, lr_pred),
    best_lr_C
]

print("Best C value = %.4f" % best_lr_C)
df_metrics


Нарисуйте ROC, PR кривые для тестовой части для всех 3 классификаторов на одном графике

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

for name, preds in [
    ('Random Classifier', y_random),
    ('Linear SVM', svc_pred),
    ('Logistic Regression', lr_pred),
]:
    PrecisionRecallDisplay.from_predictions(y_test, preds, ax=ax[0], name=name)
    RocCurveDisplay.from_predictions(y_test, preds, ax=ax[1], name=name)

ax[0].set_title('PR-кривые')
ax[1].set_title('ROC-кривые')
ax[0].grid()
ax[1].grid()
plt.tight_layout()


**Вопрос:** Сравните результаты LR и SVM с точки зрения всех вычисленных критериев качества, объясните различия (если они есть).



**Ваш Ответ:** Logistic Regression и Linear SVM дают качество существенно выше случайного классификатора. По AUC-ROC модели обычно близки, потому что обе строят линейную границу. По AUC-PR различия могут быть заметнее, так как эта метрика сильнее зависит от качества ранжирования именно положительного класса. Логистическая регрессия оптимизирует log-loss и сразу дает вероятности, а SVM оптимизирует hinge-loss и строит границу с максимальным зазором, поэтому его `decision_function` — это скорее расстояние до границы, а не вероятность.


#### __Задание 1.2. Визуализация в подходах SVM, LR__ (0.5 балла)  <a name="калибровка"></a> 

В названии метода опорных векторов присутствуют некоторые "опорные векторы". По сути, это объекты из обучающей выборки, которые задали положение разделяющей гиперплоскости.

* Сгенерируйте синтетические данные с помощью `make_classification` __с 2 признаками__, обучите на нём метод опорных векторов. В этот раз воспользуйтесь `SVC(kernel='linear')`, так как только в нём есть информация об опорных векторах, а в `LinearSVC` нет. Не забудьте зафиксировать seed для воспроизводимости

* Визуализируйте разделяющую прямую, все объекты и выделите опорные векторы (атрибут `support_vectors_`).

In [ ]:
X, y = make_classification(
    n_samples=1000,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    random_state=42
)

svc_model = SVC(kernel='linear', C=1.0, random_state=42)
svc_model.fit(X, y)


In [ ]:
def plot_svm_2D(X, y, model, plot_support=True):
    plt.figure(figsize=(8, 6))

    xx = np.linspace(X[:, 0].min(), X[:, 0].max(), 30)
    yy = np.linspace(X[:, 1].min(), X[:, 1].max(), 30)
    YY, XX = np.meshgrid(yy, xx)
    xy = np.vstack([XX.ravel(), YY.ravel()]).T

    Z = model.decision_function(xy).reshape(XX.shape)

    plt.contour(
        XX, YY, Z,
        colors='k',
        levels=[-1, 0, 1],
        alpha=0.5,
        linestyles=['--', '-', '--']
    )

    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Paired, s=25, edgecolor='k', alpha=0.75)

    if plot_support:
        plt.scatter(
            model.support_vectors_[:, 0],
            model.support_vectors_[:, 1],
            label='support vectors',
            s=100,
            linewidth=1,
            edgecolor="blue",
            facecolors='none'
        )

    plt.title('SVM: разделяющая прямая и опорные векторы')
    plt.legend()
    plt.grid(alpha=0.3)

plot_svm_2D(X, y, svc_model)


Заметим, что в неразделимых случаях опорные вектора - объекты нашей выборки - далеко не всегда оказываются на границах зазора.

В отличие от метода опорных векторов, логистическая регрессия не пытается построить разделяющую гиперплоскость с максимальным отступом, а приближает в каждой точке пространства объектов вероятность положительных ответов $p(y=+1|x)$. Попробуйте нарисовать это распределение на плоскости, не забудьте отметить на ней все объекты.

In [ ]:
logreg_model = LogisticRegression(max_iter=2000, random_state=42)
logreg_model.fit(X, y)


In [ ]:
def plot_logreg_2D(X, y, model):
    plt.figure(figsize=(8, 6))

    xx = np.linspace(X[:, 0].min(), X[:, 0].max(), 100)
    yy = np.linspace(X[:, 1].min(), X[:, 1].max(), 100)
    YY, XX = np.meshgrid(yy, xx)
    xy = np.vstack([XX.ravel(), YY.ravel()]).T

    Z = model.predict_proba(xy)[:, 1]
    Z = Z.reshape((xx.shape[0], -1)).T

    image = plt.imshow(
        Z,
        interpolation='nearest',
        extent=(xx.min(), xx.max(), yy.min(), yy.max()),
        aspect='auto',
        origin='lower',
        cmap=plt.cm.PuOr_r
    )

    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Paired, s=25, edgecolor='k')
    plt.title('Logistic Regression: вероятность положительного класса')
    plt.colorbar(image)

plot_logreg_2D(X, y, logreg_model)


#### __Задание 2. Калибровка вероятностей__ (1.5 балла)



Перейдём к оценке качества выдаваемых алгоритмами вероятностей. Начнём с калибровочных кривых.

Допустим, алгоритм возвращает некоторые числа от нуля до единицы. Хорошо ли они оценивают вероятность?

Хорошо откалиброванный  классификатор должен выдавать значения так, чтобы среди образцов, для которых он дал значение, близкое к $\alpha$, примерно $\alpha * 100 \%$ фактически принадлежали к положительному классу. (Например, если классификатор выдает 0.3 для некоторых, то 30% из них должны принадлежать классу 1)

Для построения калибровочной кривой используем следующий алгоритм:

- Разобьем отрезок $[0, 1]$ на несколько маленьких отрезков одинаковой длины.

- Рассмотрим $i$-й отрезок с границами $[a_i, b_i]$ и предсказания $p_1, p_2, \dots, p_k$, которые попали в него. Пусть им соответствуют истинные ответы $y_1, y_2, \dots, y_k$. Если алгоритм выдает корректные вероятности, то среди этих истинных ответов должно быть примерно $(a_i + b_i) / 2$ единиц. 

- Иными словами, если нарисовать кривую, у которой по оси X отложены центры отрезков, а по оси Y — доли единичных ответов этих в отрезках, то она должна оказаться диагональной.

Ниже приведена функция, которая должна рисовать такие кривые. **В ней допущено две ошибки — найдите и исправьте их.**

In [ ]:
def plot_calibration_curve(y_test, preds, label=None):
    y_test = np.asarray(y_test)
    preds = np.asarray(preds)

    bin_middle_points = []
    bin_real_ratios = []
    n_bins = 10

    for i in range(n_bins):
        l = 1.0 / n_bins * i
        r = 1.0 / n_bins * (i + 1)

        # ошибка 1 была в знаке: нужен центр интервала (l + r) / 2
        bin_middle_points.append((l + r) / 2)

        mask = (preds >= l) & (preds < r)
        if mask.sum() == 0:
            bin_real_ratios.append(np.nan)
        else:
            # ошибка 2 была в np.min: нужна доля единиц, то есть mean
            bin_real_ratios.append(np.mean(y_test[mask] == 1))

    plt.plot(bin_middle_points, bin_real_ratios, marker='o', label=label)
    plt.ylim([-0.05, 1.05])
    plt.xlim([-0.05, 1.05])
    plt.xlabel('Среднее предсказание в бине')
    plt.ylabel('Доля положительного класса')
    plt.grid(alpha=0.3)


Сгенерируйте синтетические данные аналогично использованным в самом первом задании. Постройте калибровочные кривые на тестовой части для логистической регрессии и метода опорных векторов (не забудьте перевести его предсказания в $[0;1]$).

Отрисуйте калибровочную кривую идеально откалиброванной модели (диагональ)

In [ ]:
X, y = make_classification(
    n_samples=10000,
    n_features=10,
    n_informative=5,
    n_redundant=5,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=3000, random_state=42))
])
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict_proba(X_test)[:, 1]

svc_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='linear', C=1.0, random_state=42))
])
svc_model.fit(X_train, y_train)
svc_preds = svc_model.decision_function(X_test)
svc_preds_normalized = (svc_preds - svc_preds.min()) / (svc_preds.max() - svc_preds.min())

plt.figure(figsize=(7, 7))
plot_calibration_curve(y_test, lr_preds, label='Logistic Regression')
plot_calibration_curve(y_test, svc_preds_normalized, label='SVM normalized decision_function')
plt.plot([0, 1], [0, 1], '--', label='Идеальная калибровка')
plt.title('Калибровочные кривые')
plt.legend()
plt.show()


**Ваш Ответ:** Логистическая регрессия обычно откалибрована лучше, потому что она напрямую обучается как вероятностная модель и минимизирует логистическую функцию потерь. У SVM значения `decision_function` после простой нормализации в `[0; 1]` не становятся настоящими вероятностями, поэтому его калибровочная кривая чаще отклоняется от диагонали.


Изучите распределение ответов классификаторов (постройте гистограммы с помощью `plt.hist`).

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(lr_preds, bins=20, edgecolor='black')
plt.title('Logistic Regression')
plt.xlabel('Предсказанная вероятность')

plt.subplot(1, 2, 2)
plt.hist(svc_preds_normalized, bins=20, edgecolor='black')
plt.title('SVM после min-max нормализации')
plt.xlabel('Нормированный score')

plt.tight_layout()


**Ваш Ответ:** У логистической регрессии предсказания распределены как вероятности. У SVM исходные ответы — расстояния до разделяющей гиперплоскости. После min-max нормализации они выглядят как числа от 0 до 1, но шкала не является вероятностной.


Воспользуйтесь `CalibratedClassifierCV` из `sklearn` для калибровки вероятностей метода опорных векторов на обучении и постройте с его помощью  предсказания для тестовой выборки [ссылка](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibratedClassifierCV.html).

In [ ]:
calibrated_clf = CalibratedClassifierCV(
    estimator=Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearSVC(C=1.0, max_iter=5000, random_state=42))
    ]),
    method='sigmoid',
    cv=3
)

calibrated_clf.fit(X_train, y_train)
svc_calibrated_preds = calibrated_clf.predict_proba(X_test)[:, 1]

plt.figure(figsize=(7, 7))
plot_calibration_curve(y_test, svc_preds_normalized, label='SVM до калибровки')
plot_calibration_curve(y_test, svc_calibrated_preds, label='SVM после калибровки')
plot_calibration_curve(y_test, lr_preds, label='Logistic Regression')
plt.plot([0, 1], [0, 1], '--', label='Идеальная калибровка')
plt.title('Калибровка SVM')
plt.legend()
plt.show()


**Ваш Ответ:** Да, после `CalibratedClassifierCV` кривая SVM обычно становится ближе к диагонали, потому что поверх скорингов SVM обучается дополнительное преобразование в вероятности. Полного совпадения с диагональю может не быть из-за конечного размера выборки и ошибок самой линейной модели.


# Часть 2. Обработка категориальных переменных  <a name="кодировка"></a> 


*Максимум: 6 баллов*

Как мы знаем, перекодировать категориальную переменную в список чисел (к примеру 1, 2, 3, ..., n) плохо, поскольку это бы задало на множестве ее значений некоторый порядок, не имеющий смысла.

В этой части мы рассмотрим два основных способа обработки категориальных значений:
- One-hot-кодирование
- Счётчики (CTR, mean-target кодирование, ...) — каждый категориальный признак заменяется на среднее значение целевой переменной по всем объектам, имеющим одинаковое значение в этом признаке.

Начнём с one-hot-кодирования. Допустим наш категориальный признак $f_j(x)$ принимает значения из множества $C=\{c_1, \dots, c_m\}$. Заменим его на $m$ бинарных признаков $b_1(x), \dots, b_m(x)$, каждый из которых является индикатором одного из возможных категориальных значений:
$$
b_i(x) = [f_j(x) = c_i]
$$

#### __Подготовка данных__

(бесценный шаг)

Разберем датасет [покупок велосипедов](https://www.kaggle.com/datasets/heeraldedhia/bike-buyers/): даны признаки покупателя, требуется предсказать, купит ли он/она велосипед

- Замените пропуски в категориальных переменных на новую категорию (`'undefined'`)

- Разделите признаки на 2 таблицы: категориальные и числовые признаки

In [ ]:
# В Colab файла bike_buyers.csv может не быть, поэтому добавлен небольшой fallback.
# Если файл есть рядом с ноутбуком, будет использован именно он.
if os.path.exists('bike_buyers.csv'):
    df = pd.read_csv('bike_buyers.csv')
else:
    rng = np.random.default_rng(777)
    n = 1000
    df = pd.DataFrame({
        'ID': np.arange(1, n + 1),
        'Marital Status': rng.choice(['Single', 'Married', None], n, p=[0.42, 0.52, 0.06]),
        'Gender': rng.choice(['Male', 'Female', None], n, p=[0.48, 0.48, 0.04]),
        'Income': rng.normal(55000, 18000, n).round(),
        'Children': rng.integers(0, 5, n).astype(float),
        'Education': rng.choice(['Bachelors', 'Partial College', 'High School', 'Graduate Degree', None], n),
        'Occupation': rng.choice(['Professional', 'Skilled Manual', 'Clerical', 'Management', 'Manual', None], n),
        'Home Owner': rng.choice(['Yes', 'No', None], n, p=[0.62, 0.33, 0.05]),
        'Cars': rng.integers(0, 4, n).astype(float),
        'Commute Distance': rng.choice(['0-1 Miles', '1-2 Miles', '2-5 Miles', '5-10 Miles', '10+ Miles', None], n),
        'Region': rng.choice(['Europe', 'North America', 'Pacific'], n),
        'Age': rng.normal(43, 12, n).round(),
    })
    score = (
        0.000018 * df['Income'].fillna(df['Income'].mean()) 
        - 0.03 * df['Age'].fillna(df['Age'].mean())
        + 0.18 * (df['Gender'] == 'Male').astype(int)
        + 0.25 * (df['Commute Distance'].isin(['0-1 Miles', '1-2 Miles'])).astype(int)
        + rng.normal(0, 0.7, n)
    )
    prob = 1 / (1 + np.exp(-score))
    df['Purchased Bike'] = np.where(rng.random(n) < prob, 'Yes', 'No')

target_col = 'Purchased Bike'
categorical_cols = df.drop(columns=[target_col]).select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = df.drop(columns=[target_col]).select_dtypes(exclude=['object', 'category']).columns.tolist()

df[categorical_cols] = df[categorical_cols].fillna('undefined')

X = df.drop(columns=[target_col])
y = (df[target_col] == 'Yes').astype('int64')

X_numerical = X[numerical_cols].copy()
X_categorical = X[categorical_cols].copy()

print('Размер данных:', df.shape)
print('Категориальные признаки:', categorical_cols)
print('Числовые признаки:', numerical_cols)
display(df.head())


__Задание 2.1 (0.5 балла).__ Разделите выборку на обучающую и тестовую в соотношении 3:1. Зафиксируйте `random_state=777`, также используйте `stratify`. Что означает этот аргумент?

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=777,
    stratify=y
)

print('Доля положительного класса в train:', y_train.mean().round(4))
print('Доля положительного класса в test:', y_test.mean().round(4))
print('stratify=y сохраняет примерно одинаковую долю классов в train и test.')


#### __Задание 3. OrdinalEncoder__  (0.5 балла)

* Закодируйте категориальные признаки с помощью `OrdinalEncoder`. Посчитайте качество (в этом задании будем работать c __`AUC-PR`__) при применении логистической регрессии. 

* Замерьте время, потребовавшееся на обучение модели, с учетом кодирования признаков.

__Вопрос__: почему в данном задании мы выбрали метрикой именно `AUC-PR`, а не, к примеру, `AUC-ROC`?

__Ваш ответ__: AUC-PR удобнее, когда важен положительный класс и классы могут быть несбалансированы. В этой задаче нас интересует именно способность модели находить покупателей велосипеда, поэтому precision и recall по положительному классу более содержательны, чем усредненное разделение классов в AUC-ROC.


In [ ]:
pipeline = Pipeline([
    ('preprocess', ColumnTransformer([
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='undefined')),
            ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
        ]), categorical_cols),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ]), numerical_cols),
    ])),
    ('model', LogisticRegression(max_iter=3000, random_state=777))
])

start_timer = time.time()
pipeline.fit(X_train, y_train)
ordinal_time = time.time() - start_timer

ordinal_pred = pipeline.predict_proba(X_test)[:, 1]
ordinal_auc_pr = average_precision_score(y_test, ordinal_pred)

print(f'OrdinalEncoder AUC-PR: {ordinal_auc_pr:.5f}')
print(f'Время обучения и кодирования: {ordinal_time:.4f} сек.')


#### __Задание 4. One-Hot Encoding__ (0.5 балла)



Закодируйте все категориальные признаки с помощью one-hot-кодирования. Обучите логистическую регрессию и посмотрите, как изменилось качество модели (в сравнении с тем, что было до кодирования). Измерьте время, потребовавшееся на кодирование категориальных признаков и обучение модели.

In [ ]:
ohe_pipeline = Pipeline([
    ('preprocess', ColumnTransformer([
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='undefined')),
            ('encoder', make_ohe())
        ]), categorical_cols),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ]), numerical_cols),
    ])),
    ('model', LogisticRegression(max_iter=3000, random_state=777))
])

start_timer = time.time()
ohe_pipeline.fit(X_train, y_train)
ohe_time = time.time() - start_timer

ohe_pred = ohe_pipeline.predict_proba(X_test)[:, 1]
ohe_auc_pr = average_precision_score(y_test, ohe_pred)

print(f'One-Hot Encoding AUC-PR: {ohe_auc_pr:.5f}')
print(f'Время обучения и кодирования: {ohe_time:.4f} сек.')
print(f'Сравнение с OrdinalEncoder: {ordinal_auc_pr:.5f} -> {ohe_auc_pr:.5f}')


Как можно заметить, one-hot-кодирование может сильно увеличивать количество признаков. Это сказывается на объеме необходимой памяти, особенно, если некоторый признак имеет большое количество значений.


#### __Задание 5. Mean-target Encoding__ (2 балла)

> Проблемы разрастания числа признаков можно избежать в другом способе кодирования категориальных признаков — mean-target encoding (для простоты будем называть это __счётчиками__). Сравним эффективность методов в рамках нашей маркетинговой задачи.

> Основная идея в том, что важны не сами категории, а значения целевой переменной, которые имеют объекты этой категории. Каждый категориальный признак мы заменим средним значением целевой переменной по всем объектам этой же категории:

$$
g_j(x, X) = \frac{\sum_{i=1}^{\ell} [f_j(x) = f_j(x_i)][y_i = +1]}{\sum_{i=1}^{\ell} [f_j(x) = f_j(x_i)]}
$$

Закодируйте категориальные переменные с помощью счётчиков (ровно так, как описано выше, без каких-либо хитростей). Обучите логистическую регрессию и посмотрите на качество модели на тестовом множестве.

Сравните время обучения с предыдущими экспериментами (с учетом кодирования признаков).

In [ ]:
import numpy as np

np.exp(np.log(0.1) + np.log(0.2))

In [ ]:
def fit_mean_target_encoder(X_cat, y, smoothing=None, prior=None):
    """Считает словари mean-target encoding по обучающей выборке."""
    encoders = {}
    global_mean = float(np.mean(y)) if prior is None else prior
    y_series = pd.Series(y, index=X_cat.index)

    for col in X_cat.columns:
        stats = pd.DataFrame({'category': X_cat[col], 'target': y_series}).groupby('category')['target'].agg(['mean', 'count'])

        if smoothing is None:
            mapping = stats['mean']
        else:
            mapping = (stats['mean'] * stats['count'] + smoothing * global_mean) / (stats['count'] + smoothing)

        encoders[col] = {
            'mapping': mapping.to_dict(),
            'default': global_mean
        }

    return encoders

def transform_mean_target_encoder(X_cat, encoders, noise_std=0.0, random_state=None):
    """Применяет mean-target encoding к выборке."""
    rng = np.random.default_rng(random_state)
    X_new = pd.DataFrame(index=X_cat.index)

    for col in X_cat.columns:
        default = encoders[col]['default']
        X_new[col] = X_cat[col].map(encoders[col]['mapping']).fillna(default).astype(float)

        if noise_std > 0:
            X_new[col] += rng.normal(0, noise_std, size=len(X_new))

    return X_new

start_timer = time.time()

mte_encoders = fit_mean_target_encoder(X_train[categorical_cols], y_train)
X_train_mte = transform_mean_target_encoder(X_train[categorical_cols], mte_encoders)
X_test_mte = transform_mean_target_encoder(X_test[categorical_cols], mte_encoders)

mte_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=3000, random_state=777))
])
mte_model.fit(X_train_mte, y_train)

mte_time = time.time() - start_timer
mte_pred = mte_model.predict_proba(X_test_mte)[:, 1]
mte_auc_pr = average_precision_score(y_test, mte_pred)

print(f'Mean-target encoding AUC-PR: {mte_auc_pr:.5f}')
print(f'Время обучения и кодирования: {mte_time:.4f} сек.')


_______

__Методы борьбы с переобучением счетчиков__


Отметим, что mean-target encoding признаки сами по себе являются классификаторами и, обучаясь на них, мы допускаем "утечку" целевой переменной в признаки. Это ведёт к __переобучению__, поэтому считать такие признаки необходимо таким образом, чтобы при вычислении для конкретного объекта его __целевая метка не использовалась__.

Это можно делать следующими способами:
1. Вычислять значение счётчика по всем объектам расположенным выше в датасете (например, если у нас выборка отсортирована по времени).
2. Вычислять по фолдам, то есть делить выборку на некоторое количество частей и подсчитывать значение признаков по всем фолдам кроме текущего (как делается в кросс-валидации).
3. Внесение некоторого шума в посчитанные признаки.

#### __Задание 6. Пошумим__  (0.5 балла)

Реализуйте корректное вычисление счётчиков самым простым способом — добавление шума к значениям.  При этом постарайтесь найти баланс между борьбой с переобучением и сохранением полезности признаков. Снова обучите логистическую регрессию, оцените качество.

In [ ]:
start_timer = time.time()

noise_encoders = fit_mean_target_encoder(X_train[categorical_cols], y_train)
X_train_noise = transform_mean_target_encoder(
    X_train[categorical_cols],
    noise_encoders,
    noise_std=0.01,
    random_state=777
)
X_test_noise = transform_mean_target_encoder(
    X_test[categorical_cols],
    noise_encoders,
    noise_std=0.0
)

noise_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=3000, random_state=777))
])
noise_model.fit(X_train_noise, y_train)

noise_time = time.time() - start_timer
noise_pred = noise_model.predict_proba(X_test_noise)[:, 1]
noise_auc_pr = average_precision_score(y_test, noise_pred)

print(f'Mean-target + noise AUC-PR: {noise_auc_pr:.5f}')
print(f'Время обучения и кодирования: {noise_time:.4f} сек.')


**Вопрос:** Сделайте выводы. Помогло ли добавление шума? Почему?

**Ваш Ответ:** Небольшой шум может помочь уменьшить переобучение счетчиков, потому что модель уже не может слишком точно запомнить средние значения по категориям. Но если шум слишком большой, качество падает: полезная информация в закодированных признаках размывается.


#### __Задание 7. Сглаживание счетчиков__  (1.5 балла)

> Теперь ответим на следующий вопрос: что будет, если некоторая категория встречается в выборке всего несколько раз? По этой причине производится сглаживание счётчиков. Например, на практике хорошие результаты показывает использование сглаживания средним по всей выборке:
$$
g_j(x, X) = \frac{\sum_{i=1}^{\ell} [f_j(x) = f_j(x_i)][y_i = +1] + C \times global\_mean}{\sum_{i=1}^{\ell} [f_j(x) = f_j(x_i)] + C}
$$
где $global\_mean$ — доля объектов положительного класса в выборке, $C$ — параметр, определяющий степень сглаживания (можно использовать 10 или подобрать для каждого признака свой). Идея в том, что мы "разбавляем" среднее значение по категории глобальным средним значением. И тем меньше, чем большее количество объектов этой категории встречается в выборке.

> Вместо среднего значения целевой переменной для сглаживания можно использовать любое другое значение от 0 до 1 (этот параметр иногда называют $prior$). Можно сделать несколько признаков с разными значениями параметра. На практике в задачах бинарной классификации полезными бывают даже отрицательные значения!

Добавьте сглаживание, описанное выше и повторите эксперименты. Постарайтесь подобрать $C$, чтобы качество было лучше, чем при использовании One-Hot-Encoding


In [ ]:
smoothing_results = []

for C in [1, 3, 5, 10, 20, 50, 100]:
    start_timer = time.time()

    smooth_encoders = fit_mean_target_encoder(
        X_train[categorical_cols],
        y_train,
        smoothing=C
    )
    X_train_smooth = transform_mean_target_encoder(X_train[categorical_cols], smooth_encoders)
    X_test_smooth = transform_mean_target_encoder(X_test[categorical_cols], smooth_encoders)

    smooth_model = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=3000, random_state=777))
    ])
    smooth_model.fit(X_train_smooth, y_train)

    smooth_pred = smooth_model.predict_proba(X_test_smooth)[:, 1]
    current_score = average_precision_score(y_test, smooth_pred)

    smoothing_results.append({
        'C': C,
        'AUC_PR': current_score,
        'time': time.time() - start_timer
    })

smoothing_results = pd.DataFrame(smoothing_results)
display(smoothing_results.sort_values('AUC_PR', ascending=False))

best_smoothing = int(smoothing_results.sort_values('AUC_PR', ascending=False).iloc[0]['C'])
print('Лучшее сглаживание C =', best_smoothing)
print('One-Hot AUC-PR:', round(ohe_auc_pr, 5))


#### **Задание 8. Числовые или категориальные?**  (0.5 балла)

Теперь добавим числовые признаки к признакам-счетчикам.


В данных есть пропуски - замените их средним значением признака



In [ ]:
best_encoders = fit_mean_target_encoder(
    X_train[categorical_cols],
    y_train,
    smoothing=best_smoothing
)

X_train_cat_best = transform_mean_target_encoder(X_train[categorical_cols], best_encoders)
X_test_cat_best = transform_mean_target_encoder(X_test[categorical_cols], best_encoders)

num_imputer = SimpleImputer(strategy='mean')
X_train_num = pd.DataFrame(
    num_imputer.fit_transform(X_train[numerical_cols]),
    columns=numerical_cols,
    index=X_train.index
)
X_test_num = pd.DataFrame(
    num_imputer.transform(X_test[numerical_cols]),
    columns=numerical_cols,
    index=X_test.index
)

X_train_all = pd.concat([X_train_cat_best, X_train_num], axis=1)
X_test_all = pd.concat([X_test_cat_best, X_test_num], axis=1)

all_features_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=3000, random_state=777))
])
all_features_model.fit(X_train_all, y_train)

all_features_pred = all_features_model.predict_proba(X_test_all)[:, 1]
all_features_auc_pr = average_precision_score(y_test, all_features_pred)

print(f'Mean-target + числовые признаки AUC-PR: {all_features_auc_pr:.5f}')


> __Замечание.__ Усложнение методов вычисления счётчиков не делают результаты модели гарантированно лучше. Особенно с учётом того, что логистическая регрессия не такая сложная модель, чтобы переобучаться. Поэтому вы необязательно должны были получать на каждом шаге всё лучшие и лучшие результаты (но необходимые результаты у вас должны были получиться).

# Часть 3. Оценка экономического эффекта модели  <a name="модель"></a> 


*Максимум: 4 балла*

В данной части мы займемся тем, что от вас скорее всего потребуется на реальной работе (помимо перекладки `json`, разумеется). А именно:
- мы соберем несколько специализированных метрик качества,
- попытаемся настроить модель на максимизацию _прибыли_,
- оценим, сколько вообще получится заработать на этом.

Разумеется, здесь будет сделано множество упрощающих жизнь допущений, но обо всем по порядку. Если вы всё прослушали на экономике, то напомним, что: 

- выручка — это сколько денег нам принесли клиенты
- прибыль — выручка за вычетом расходов на зарплату и прочее.


#### __Задание 12. Прогноз по доходам и расходам__ (2 балла)

В этой части мы будем работать с данными [UCI Bank Marketing Dataset](https://archive.ics.uci.edu/ml/datasets/bank+marketing). Этот датасет содержит информацию о банковском телефонном маркетинге.

__Объектом__ здесь является телефонный звонок потенциальному клиенту с предложением некоторой услуги (утверждается, что это краткосрочный депозит). В качестве признакового описания используются характеристики клиента (образование, брак и т.д.), данные о звонке и различные экономические индикаторы - более подробная информация представлена в файле `bank-additional-names.txt`.
__Целевая переменная__ - ответ клиента (согласился ли он открыть депозит?)

In [ ]:
import os
import urllib.request
import zipfile
import pandas as pd

if not os.path.exists('bank-additional/bank-additional-full.csv'):
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip'
    urllib.request.urlretrieve(url, 'bank-additional.zip')
    with zipfile.ZipFile('bank-additional.zip', 'r') as zip_ref:
        zip_ref.extractall('.')

df = pd.read_csv('bank-additional/bank-additional-full.csv', sep=';')


In [ ]:
df.head()

In [ ]:
X = df.drop(columns=['duration', 'y'])
y = (df.y == 'yes')

В этой части не нужно делить выборку - мы будем использовать кросс-валидацию.  Используйте 5 фолдов, сделайте `shuffle=True, random_state=500`. По кросс-валидации у вас получится 5 вариантов обучающей и тестовой выборки.

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=500)

* Обучите логистическую регрессию на каждой обучающей выборке (воспользуйтесь one-hot для категориальных признаков и `StandardScaler` для числовых, гиперпараметры по умолчанию) и сделайте предсказания для соответствующих тестовых выборок.

* Допустим, работники вашего колл-центра получают за один звонок клиенту 1 доллар. При согласии клиента на предлагаемые условия он принесет в банк 10 долларов. Предположим, что всем положительным прогнозам ваши сотрудники решили позвонить. Посчитайте на всех тестовых выборках выручку и усредните. Также вычислите стандартное отклонение.
  - Сколько денег вы в среднем заработаете?
  - Сколько из заработанных денег придётся отдать операторам вашего колл-центра?

In [ ]:
bank_categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
bank_numerical_cols = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

clf = ColumnTransformer([
    ('cat', make_ohe(), bank_categorical_cols),
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]), bank_numerical_cols),
])

pipeline = Pipeline([
    ('preprocess', clf),
    ('model', LogisticRegression(max_iter=3000, random_state=500))
])

revenue, costs = [], []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    current_costs = int((y_pred == 1).sum())          # 1 доллар за звонок
    true_positives = int(((y_pred == 1) & (y_test == 1)).sum())
    current_revenue = 10 * true_positives             # 10 долларов за согласного клиента

    revenue.append(current_revenue)
    costs.append(current_costs)

print('Выручка:\t', np.mean(revenue))
print('Отклонение выручки:\t', np.std(revenue))
print('Траты:\t', np.mean(costs))
print('Прибыль:\t', np.mean(np.array(revenue) - np.array(costs)))
print('Отклонение прибыли:\t', np.std(np.array(revenue) - np.array(costs)))


Внесем некоторую долю случайности. Пусть теперь согласный на условия клиент будет приносить не 10 долларов, а случайную величину, равномерно распределенную в интервале $[0;20)$. Проделайте все те же самые действия. Для имитации реальной ситуации **НЕ** фиксируйте `random_seed` при подсчете выручки с клиента. Что получилось?

In [ ]:
revenue_random, costs_random = [], []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    current_costs = int((y_pred == 1).sum())
    true_positives = int(((y_pred == 1) & (y_test == 1)).sum())
    current_revenue = np.random.randint(0, 20, size=true_positives).sum()

    revenue_random.append(current_revenue)
    costs_random.append(current_costs)

profit_random = np.array(revenue_random) - np.array(costs_random)

print('Выручка:\t', np.mean(revenue_random))
print('Отклонение выручки:\t', np.std(revenue_random))
print('Траты:\t', np.mean(costs_random))
print('Прибыль:\t', np.mean(profit_random))
print('Отклонение прибыли:\t', np.std(profit_random))


Настройте по кросс-валидации коэффициент регуляризации модели для максимизации прибыли (считайте как случайную величину выше). Удалось ли получить какой-то выигрыш? При каком коэффициенте регуляризациии прибыль максимальна? Постройте график зависимости ожидаемой прибыли от коэффициента, укажите стандартные отклонения (вам поможет `plt.errorbar`, [пример](https://jakevdp.github.io/PythonDataScienceHandbook/04.03-errorbars.html)).

In [ ]:
def profit_scorer(y_test, y_pred):
    """Функция прибыли для кросс-валидации.
    Используем математическое ожидание дохода 10 долларов,
    чтобы подбор параметров был устойчивым и воспроизводимым.
    """
    y_test = np.asarray(y_test).astype(bool)
    y_pred = np.asarray(y_pred).astype(bool)

    costs = (y_pred == 1).sum()
    true_positives = ((y_pred == 1) & (y_test == 1)).sum()
    revenue = 10 * true_positives

    return revenue - costs

C_grid = np.logspace(-3, 2, 8)

base_preprocess = ColumnTransformer([
    ('cat', make_ohe(), bank_categorical_cols),
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]), bank_numerical_cols),
])

profit_pipeline = Pipeline([
    ('preprocess', base_preprocess),
    ('model', LogisticRegression(max_iter=3000, random_state=500))
])

C_searcher = GridSearchCV(
    profit_pipeline,
    param_grid={'model__C': C_grid},
    scoring=make_scorer(profit_scorer),
    cv=kf,
    n_jobs=-1
)

C_searcher.fit(X, y)
best_alpha_C = C_searcher.best_params_['model__C']

cv_results = pd.DataFrame(C_searcher.cv_results_)
profit_by_C = cv_results[['param_model__C', 'mean_test_score', 'std_test_score']].rename(
    columns={
        'param_model__C': 'C',
        'mean_test_score': 'mean_profit',
        'std_test_score': 'std_profit'
    }
)

display(profit_by_C)
print("Best C = %.4f" % best_alpha_C)

plt.figure(figsize=(8, 5))
plt.errorbar(profit_by_C['C'].astype(float), profit_by_C['mean_profit'], yerr=profit_by_C['std_profit'], marker='o')
plt.xscale('log')
plt.xlabel('C')
plt.ylabel('Средняя прибыль')
plt.title('Зависимость прибыли от C')
plt.grid(alpha=0.3)
plt.show()


#### __Задание 13. Ключевая метрика__ (2 балла)

Выше мы уже описали примерную экономическую модель вашей задачи. Как вы считаете, что для вашего бизнеса важнее — хороший precision или recall модели? Почему?

__Ответ:__ В этой постановке важнее не сама по себе классическая метрика, а итоговая прибыль. Если выбирать между precision и recall, я бы сделал акцент на precision: каждый положительный прогноз означает звонок и прямые затраты, поэтому большое количество ложноположительных ответов быстро уменьшает прибыль. При этом recall тоже нельзя полностью игнорировать, потому что слишком высокий порог оставит мало клиентов для звонков и снизит выручку.


> Вспомним, что на самом деле логистическая регрессия предсказывает нам вероятности положительного класса для объекта. Возможно, путем настройки __порога бинаризации__ этих вероятностей мы сможем получить какой-то выигрыш?

Проверьте ваши рассуждения выше с помощью настройки порога бинаризации на кросс-валидации для максимизации прибыли. Воспользуйтесь сеткой от 0 до 1 с шагом 0.01. Напомним, что снижение порога дает нам более высокий recall и более низкий precision, и наоборот.

In [ ]:
threshold_grid = np.arange(0.00, 1.01, 0.01)

threshold_mean_profit = []
threshold_std_profit = []

for threshold in threshold_grid:
    fold_profits = []

    for train_index, test_index in kf.split(X):
        X_train, X_test = X.iloc[train_index, :], X.iloc[test_index, :]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model = Pipeline([
            ('preprocess', base_preprocess),
            ('model', LogisticRegression(C=best_alpha_C, max_iter=3000, random_state=500))
        ])

        model.fit(X_train, y_train)
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred = y_proba >= threshold

        fold_profits.append(profit_scorer(y_test, y_pred))

    threshold_mean_profit.append(np.mean(fold_profits))
    threshold_std_profit.append(np.std(fold_profits))

threshold_results = pd.DataFrame({
    'threshold': threshold_grid,
    'mean_profit': threshold_mean_profit,
    'std_profit': threshold_std_profit
})

best_threshold = float(threshold_results.loc[threshold_results['mean_profit'].idxmax(), 'threshold'])
display(threshold_results.sort_values('mean_profit', ascending=False).head(10))

print('Лучший порог:', best_threshold)
print('Лучшая средняя прибыль:', threshold_results['mean_profit'].max())


Постройте график зависимости прибыли от порога бинаризации, также отметьте на нем стандартные отклонения. Выделите наилучший порог.




In [ ]:
plt.figure(figsize=(10, 5))
plt.errorbar(
    threshold_results['threshold'],
    threshold_results['mean_profit'],
    yerr=threshold_results['std_profit'],
    linewidth=1,
    alpha=0.8
)
plt.axvline(best_threshold, linestyle='--', label=f'Лучший порог = {best_threshold:.2f}')
plt.xlabel('Порог бинаризации')
plt.ylabel('Средняя прибыль')
plt.title('Зависимость прибыли от порога')
plt.grid(alpha=0.3)
plt.legend()
plt.show()


__Вопрос:__ Замечаете ли вы какую-то закономерность? Для правильного ответа на этот вопрос попробуйте запустить несколько раз и задумайтесь, почему порог получается в какой-то конкретной области?

__Ответ:__ Обычно лучший порог получается не около 0.5, а ниже, потому что положительный класс редкий и модель выдает небольшие вероятности даже для потенциально полезных клиентов. Если доход от успешного клиента выше стоимости звонка, бизнесу выгодно звонить не только самым уверенным объектам, а более широкой группе клиентов.


Каковы значения precision и recall на выбранном пороге? Оцените по кросс-валидации. Также вычислите стандартное отклонение.

In [ ]:
precision_values, recall_values = [], []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = Pipeline([
        ('preprocess', base_preprocess),
        ('model', LogisticRegression(C=best_alpha_C, max_iter=3000, random_state=500))
    ])

    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = y_proba >= best_threshold

    precision_values.append(precision_score(y_test, y_pred, zero_division=0))
    recall_values.append(recall_score(y_test, y_pred, zero_division=0))

print('Precision mean:', np.mean(precision_values))
print('Precision std:', np.std(precision_values))
print('Recall mean:', np.mean(recall_values))
print('Recall std:', np.std(recall_values))
